# 🏗️ Notebook 1: Google Calendar — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/google-calendar
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A calendar service. Users create events (optionally recurring), invite guests, share calendars, and get reminders. Timezones and recurrence are the tricky parts.

## Requirements

### Functional
- CRUD events (one-off or recurring via RRULE).
- Invite guests; RSVP.
- Share calendars (read/busy/write).
- Reminders (push / email).

### Non-functional
- Correct timezone handling (DST).
- Fan-out reminders at exact times.
- Eventual consistency across replicas.

## Back-of-envelope

- 1B users × 20 events/week → 20B events/year.
- Reminder firings: 100k/s peak — needs a durable scheduler (see `reminder-alert` lab).

## High-level architecture

```
  [Client]
     │
     ▼
  API Gateway
     │
   ┌─┴────────────┬────────────┬──────────────┐
   ▼              ▼            ▼              ▼
 Event Svc    Calendar Svc  Reminder Svc   Sharing/ACL Svc
   │              │            │              │
   ▼              ▼            ▼              ▼
  PG             PG         Scheduled-Q      PG
```

- **Event Service** stores event rows; for recurring events it stores the RRULE, not every occurrence.
- **Occurrences** are expanded on read within the query window.

## Why these choices?

- Each service in the diagram owns one responsibility — easier to scale and reason about.
- Stateless services scale horizontally; stateful stores are chosen per access pattern.
- The next two notebooks zoom into the **data model + APIs** and one **deep-dive algorithm**.